In [11]:
from langchain.agents import tool,create_react_agent,AgentExecutor
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from langchain import hub
from langchain_core.messages import SystemMessage
from langchain_core.prompts.chat import ChatPromptTemplate,MessagesPlaceholder
from pprint import pprint
import requests
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")
TMDB_API_KEY = os.getenv("TMDB_API_KEY")
OMDB_API_KEY = os.getenv("OMDB_API_KEY")

In [12]:
def get_movie_details(title):
    """Fetches movie details including plot, genre, and IMDb rating using the OMDb API."""
    url = "http://www.omdbapi.com/"
    params = {
        "apikey": OMDB_API_KEY,
        "t": title
    }

    response = requests.get(url, params=params)
    data = response.json()
    
    if data.get("Response") == "False":
        return f"Movie '{title}' not found."

    # Extract required details
    title = data.get("Title", "Title not available.")
    year = data.get("Year", "Year not available.")
    genre = data.get("Genre", "Genre not available.")
    plot = data.get("Plot", "Plot details not available.")
    imdb_rating = data.get("imdbRating", "Rating not available.")

    # Return dict so LLM can format
    return {
        "Title": title,
        "Year": year,
        "Genre": genre,
        "IMDb Rating": imdb_rating,
        "Plot": plot
    }

#Tool
@tool
def fetch_movie_plot(input_str: str):
    """Fetch the plot details of a given movie title."""
    return get_movie_details(input_str.strip())

In [13]:
tools = [fetch_movie_plot]

from langchain_core.prompts import PromptTemplate

template = """Answer the following questions with movie details.
You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: Provide the response in this format:

Title: <Movie Title>
Year: <Year>
Genre: <Genre>
IMDb Rating: <Rating>
Plot: <Plot>

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)


# initialize llm
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.7,
    google_api_key=api_key,
)

In [14]:
# initialize an agent

agent = create_react_agent(
    llm,
    tools,
    prompt
)

agent_executor = AgentExecutor(agent = agent,tools = tools)

In [15]:
response = agent_executor.invoke({"input": "What is the story of movie EEGA"})
pprint(response)

{'input': 'What is the story of movie EEGA',
 'output': 'Title: Eega\n'
           'Year: 2012\n'
           'Genre: Action, Comedy, Drama\n'
           'IMDb Rating: 7.7\n'
           'Plot: Nani loves Bindu but is killed by a jealous Sudeep, who '
           'lusts after Bindu. Nani is reincarnated as a fly and decides to '
           "avenge his death. He teams up with Bindu to make Sudeep's life a "
           'living hell.'}
